# Como foi feita a app comercial, passo a passo

**A pergunta de negócio.** O diretor comercial de um distribuidor português de saúde animal quer saber onde a
empresa está a perder faturação, margem e tesouraria, e o que fazer esta semana. A app responde a cinco perguntas:

1. Que região está a cair, e porquê?
2. Quem está a oferecer margem?
3. Que clientes devemos contactar esta semana?
4. Onde está o dinheiro parado?
5. O que é sazonal?

**Como está organizado este notebook.** Segue o projeto pela ordem em que foi construído:

| Passo | O quê | Ferramenta |
|---|---|---|
| 1 | Gerar um conjunto de dados realista com regras de negócio documentadas | Python (NumPy) |
| 2 | Carregar e verificar os ficheiros em bruto | pandas |
| 3 | Modelar um esquema em estrela com testes | dbt sobre DuckDB |
| 4 | Responder às cinco perguntas com gráficos interativos | pandas + Altair |
| 5 | Conferir o pandas com os marts do dbt | pandas + DuckDB |
| 6 | Publicar: painel web, Power BI e Qlik | Exportação JSON, PBIP, script Qlik |

Os dados são **sintéticos**: um gerador com semente fixa escreve 1800 clientes, 12 vendedores e cerca de 951 000 linhas de
encomenda, de janeiro de 2024 a setembro de 2026. As regras que segue estão listadas no passo 1, para que quem lê possa
confirmar que a análise as encontra só a partir dos dados.

**Os gráficos são interativos**: passe o cursor para ver os valores; clique numa barra, num ponto ou numa entrada da legenda para
filtrar o gráfico ligado. São feitos com [Altair](https://altair-viz.github.io/). O GitHub mostra os notebooks sem JavaScript,
por isso abra a [versão interativa](https://guilhermerigonato.com/commercial-analytics/notebook/pt/) para os usar.
O código e os nomes das colunas ficam em inglês, como no repositório.

**Para correr:** `pip install pandas numpy altair duckdb dbt-duckdb`; depois, na raiz do repositório,
`python generator/generate.py` e `dbt build --profiles-dir .`, e execute todas as células (cerca de 1 minuto).

In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import altair as alt

DATA = Path("../data")
AS_OF = pd.Timestamp("2026-09-30")          # data de referência, a variável `as_of` do dbt
pd.set_option("display.float_format", "{:,.2f}".format)

BLUE, ORANGE, GREEN, GREY, MUTED, GRID = "#2a78d6", "#eb6834", "#1baf7a", "#c9c8c3", "#52514e", "#e4e3df"

@alt.theme.register("portfolio", enable=True)
def portfolio_theme():
    axis = {"gridColor": GRID, "domainColor": MUTED, "tickColor": MUTED, "labelColor": MUTED,
            "titleColor": MUTED, "labelFontSize": 11, "titleFontSize": 11, "titleFontWeight": "normal"}
    return {"config": {"view": {"stroke": None}, "axis": axis, "legend": {"labelColor": MUTED, "titleColor": MUTED},
                       "title": {"anchor": "start", "fontSize": 13, "color": "#0b0b0b"},
                       "font": "system-ui, -apple-system, Segoe UI, sans-serif"}}

## Passo 1. Gerar os dados

Os dados reais de clientes de um distribuidor não podem ser publicados, por isso o projeto parte de um gerador
([`generator/generate.py`](../generator/generate.py)) que escreve ficheiros CSV com comportamentos de negócio **plantados de
propósito**. As regras, tal como estão na docstring do gerador:

| Regra | Comportamento | Pergunta que testa |
|---|---|---|
| R1 | Os clientes encomendam mais nas semanas a seguir a uma visita do vendedor | 3 |
| R2 | Os clientes sem visita há 60 ou mais dias começam a deixar de comprar | 3 |
| R3 | O Sul perdeu um vendedor em janeiro de 2026; os seus clientes passaram a ser visitados metade das vezes | 1 |
| R4 | O vendedor REP07 dá cerca de 9 pontos de desconto a mais do que os outros, sem ganho de volume | 2 |
| R5 | As explorações pecuárias pagam tarde; cerca de 7% das suas faturas atrasam 90 a 200 dias | 4 |
| R6 | As vacinas têm pico na primavera, os antiparasitários no verão, a alimentação é estável | 5 |

A semente fixa (`20261024`) torna todas as execuções idênticas, por isso os números deste notebook, os testes do dbt e os
painéis coincidem. A geração demora cerca de 40 segundos.

In [2]:
# Correr uma vez na raiz do repositório:  python generator/generate.py
files = sorted(DATA.glob("*.csv"))
pd.DataFrame({"file": [f.name for f in files],
              "rows": [sum(1 for _ in open(f, encoding="utf-8")) - 1 for f in files],
              "MB": [round(f.stat().st_size / 1e6, 1) for f in files]})

,file,rows,MB
0,customers.csv,1800,0.10
1,invoices.csv,130589,6.30
2,order_lines.csv,951489,35.40
3,orders.csv,130589,4.30
4,payments.csv,125000,3.50
5,products.csv,400,0.00
6,reps.csv,12,0.00
7,visits.csv,67836,2.20


## Passo 2. Carregar e verificar os ficheiros em bruto

Os ficheiros vêm normalizados como numa exportação de ERP: encomendas e linhas de encomenda estão separadas, a categoria do produto
está na tabela de produtos e os pagamentos são um ficheiro à parte, ligado pela fatura. Antes de modelar, três verificações que
apanham a maioria dos problemas de exportação: as chaves são únicas, todas as chaves estrangeiras existem e as datas estão dentro do período.

In [3]:
customers = pd.read_csv(DATA / "customers.csv")
reps = pd.read_csv(DATA / "reps.csv")
products = pd.read_csv(DATA / "products.csv")
orders = pd.read_csv(DATA / "orders.csv", parse_dates=["order_date"])
lines = pd.read_csv(DATA / "order_lines.csv")
visits = pd.read_csv(DATA / "visits.csv", parse_dates=["visit_date"])
invoices = pd.read_csv(DATA / "invoices.csv", parse_dates=["invoice_date", "due_date"])
payments = pd.read_csv(DATA / "payments.csv", parse_dates=["payment_date"])

checks = {
    "customer_id unique": customers["customer_id"].is_unique,
    "order_id unique": orders["order_id"].is_unique,
    "invoice_id unique": invoices["invoice_id"].is_unique,
    "every line has an order": lines["order_id"].isin(orders["order_id"]).all(),
    "every line has a product": lines["sku"].isin(products["sku"]).all(),
    "every order has a customer": orders["customer_id"].isin(customers["customer_id"]).all(),
    "every payment has an invoice": payments["invoice_id"].isin(invoices["invoice_id"]).all(),
    "orders inside Jan 2024–Sep 2026": orders["order_date"].between("2024-01-01", AS_OF).all(),
    "net price never above list price": (lines["net_price_eur"] <= lines["list_price_eur"] + 1e-9).all(),
}
assert all(checks.values()), checks
pd.Series(checks, name="passes").to_frame()

,passes
customer_id unique,True
order_id unique,True
invoice_id unique,True
every line has an order,True
every line has a product,True
every order has a customer,True
every payment has an invoice,True
orders inside Jan 2024–Sep 2026,True
net price never above list price,True


## Passo 3. Modelar um esquema em estrela em dbt

O SQL está em [`models/`](../models). As views de staging tipam e renomeiam as colunas dos CSV; os marts formam um esquema
em estrela, mais um ou dois marts por pergunta:

* **Factos:** `fct_sales_lines` (grão da linha de encomenda, valores bruto / líquido / custo), `fct_visits`, `fct_invoices`
  (uma linha por fatura, com o pagamento e o escalão de antiguidade na data de referência).
* **Dimensões:** `dim_customer` (com indicadores de recência na data de referência), `dim_product`, `dim_rep`, `dim_date`
  (com indicadores de acumulado do ano e últimos 12 meses face à data de referência).
* **Marts por pergunta:** `mart_region_ytd`, `mart_rep_pricing`, `mart_call_list`, `mart_receivables_aging`,
  `mart_payment_by_segment`, `mart_category_seasons` e outros.

`dbt build --profiles-dir .` corre os modelos e os testes em conjunto: chaves únicas e não nulas, relações
entre factos e dimensões, valores aceites para segmentos e escalões, e verificações de que os marts somam o mesmo que os
factos. Com estes dados: **87 nós, 0 falhas**. O modelo escreve `data/commercial.duckdb`, que o passo 5 lê.

Em pandas, o facto das linhas de encomenda fica a um merge de distância:

In [4]:
sales = (lines.merge(orders, on="order_id").merge(products[["sku", "category"]], on="sku")
              .merge(customers[["customer_id", "region", "segment"]], on="customer_id"))
sales["gross"] = sales["quantity"] * sales["list_price_eur"]
sales["net"] = sales["quantity"] * sales["net_price_eur"]
sales["cost"] = sales["quantity"] * sales["unit_cost_eur"]
sales["month"] = sales["order_date"].dt.to_period("M").dt.to_timestamp()
visits = visits.merge(customers[["customer_id", "region"]], on="customer_id")
visits["month"] = visits["visit_date"].dt.to_period("M").dt.to_timestamp()

ytd = sales["order_date"].between("2026-01-01", AS_OF)
pytd = sales["order_date"].between("2025-01-01", "2025-09-30")
print(f"{len(sales):,} linhas de encomenda · faturação líquida €{sales['net'].sum() / 1e6:,.1f}m · "
      f"jan–set 2026 €{sales.loc[ytd, 'net'].sum() / 1e6:,.1f}m vs €{sales.loc[pytd, 'net'].sum() / 1e6:,.1f}m um ano antes")

951 489 linhas de encomenda · faturação líquida 121,3 M€ · jan–set 2026 34,9 M€ vs 33,7 M€ um ano antes


## Passo 4. Responder às cinco perguntas

### P1. Que região está a cair, e porquê?

Compare janeiro–setembro de 2026 com os mesmos meses de 2025, na faturação e nas visitas dos vendedores.
**Clique numa região** no gráfico da esquerda para ver a faturação e as visitas mensais à direita; duplo clique para limpar.

In [5]:
def yoy(frame, value, date_col):
    cur = frame[frame[date_col].between("2026-01-01", AS_OF)].groupby("region")[value].sum()
    prev = frame[frame[date_col].between("2025-01-01", "2025-09-30")].groupby("region")[value].sum()
    return cur, prev

rev_cur, rev_prev = yoy(sales, "net", "order_date")
visits["n"] = 1
vis_cur, vis_prev = yoy(visits, "n", "visit_date")
q1 = pd.DataFrame({"revenue_2026": rev_cur, "revenue_2025": rev_prev, "visits_2026": vis_cur, "visits_2025": vis_prev})
q1["revenue_change"] = q1["revenue_2026"] / q1["revenue_2025"] - 1
q1["visits_change"] = q1["visits_2026"] / q1["visits_2025"] - 1
q1 = q1.sort_values("revenue_change").reset_index()
q1

,region,revenue_2026,revenue_2025,visits_2026,visits_2025,revenue_change,visits_change
0,Sul,"3,824,993.42","4,578,433.63",1426,2722,-0.16,-0.48
1,Lisboa e Vale do Tejo,"11,882,545.85","11,311,654.79",6923,6539,0.05,0.06
2,Norte,"10,817,645.33","10,224,122.32",5682,5390,0.06,0.05
3,Centro,"8,399,395.26","7,607,882.40",4500,4189,0.10,0.07


In [6]:
monthly = (sales.groupby(["region", "month"])["net"].sum().rename("revenue").reset_index()
                .merge(visits.groupby(["region", "month"]).size().rename("visits").reset_index(), on=["region", "month"]))

pick = alt.selection_point(fields=["region"], on="click", clear="dblclick", empty=True)
bars = alt.Chart(q1).mark_bar(cornerRadius=3).encode(
    y=alt.Y("region:N", sort=list(q1["region"]), title=None),
    x=alt.X("revenue_change:Q", title="Faturação líquida, jan–set 2026 vs 2025", axis=alt.Axis(format="+%")),
    color=alt.condition("datum.revenue_change < 0", alt.value(ORANGE), alt.value(BLUE)),
    opacity=alt.condition(pick, alt.value(1), alt.value(0.3)),
    tooltip=[alt.Tooltip("region:N", title="Região"),
             alt.Tooltip("revenue_change:Q", title="Faturação", format="+.1%"),
             alt.Tooltip("visits_change:Q", title="Visitas", format="+.1%")],
).add_params(pick).properties(width=260, height=180, title="Clique numa região")

base = alt.Chart(monthly).transform_filter(pick).transform_aggregate(
    revenue="sum(revenue)", visits="sum(visits)", groupby=["month"])
rev_line = base.mark_line(color=BLUE, strokeWidth=2.5).encode(
    x=alt.X("month:T", title=None), y=alt.Y("revenue:Q", title="Faturação líquida (€)", axis=alt.Axis(format="~s")),
    tooltip=[alt.Tooltip("month:T", format="%b %Y"), alt.Tooltip("revenue:Q", format=",.0f"), alt.Tooltip("visits:Q")])
vis_bar = base.mark_bar(color="#a8c8ef", opacity=0.7).encode(
    x="month:T", y=alt.Y("visits:Q", title="Visitas dos vendedores"))
rule = alt.Chart(pd.DataFrame({"d": [pd.Timestamp("2026-01-01")]})).mark_rule(strokeDash=[3, 3], color=MUTED).encode(x="d:T")
right = alt.layer(vis_bar, rev_line, rule).resolve_scale(y="independent").properties(
    width=420, height=180, title="Faturação mensal (linha) e visitas (barras); tracejado: saída do REP12 no Sul")
alt.hconcat(bars, right)

alt.HConcatChart(...)

**Resposta.** O Sul é a única região em queda (−16,5%, contra +5% a +10% nas outras), e as visitas quase caíram para metade
(−48%) depois de o REP12 sair em janeiro de 2026: a regra R3, encontrada a partir dos dados. A queda aparece primeiro nas visitas e a
faturação vem atrás, o que aponta para cobertura comercial, não para preço nem procura.

### P2. Quem está a oferecer margem?

Desconto = 1 − vendas líquidas ÷ vendas a preço de tabela; margem bruta = (líquido − custo) ÷ líquido. **Clique num vendedor** para
ver o seu desconto por categoria à direita.

In [7]:
cur = sales[ytd]
by_rep = cur.groupby("rep_id")[["gross", "net", "cost"]].sum()
by_rep["discount"] = 1 - by_rep["net"] / by_rep["gross"]
by_rep["margin"] = (by_rep["net"] - by_rep["cost"]) / by_rep["net"]
by_rep = by_rep.join(reps.set_index("rep_id")["region"]).reset_index()

by_rep_cat = cur.groupby(["rep_id", "category"])[["gross", "net"]].sum().reset_index()
by_rep_cat["discount"] = 1 - by_rep_cat["net"] / by_rep_cat["gross"]

pick_rep = alt.selection_point(fields=["rep_id"], on="click", clear="dblclick", empty=False, value=[{"rep_id": "REP07"}])
scatter = alt.Chart(by_rep).mark_circle(stroke="white", strokeWidth=1).encode(
    x=alt.X("discount:Q", title="Desconto sobre o preço de tabela", axis=alt.Axis(format="%"), scale=alt.Scale(zero=False)),
    y=alt.Y("margin:Q", title="Margem bruta", axis=alt.Axis(format="%"), scale=alt.Scale(zero=False)),
    size=alt.Size("net:Q", legend=None, scale=alt.Scale(range=[80, 500])),
    color=alt.condition(pick_rep, alt.value(ORANGE), alt.value(BLUE)),
    tooltip=["rep_id", "region", alt.Tooltip("net:Q", title="Faturação líquida", format=",.0f"),
             alt.Tooltip("discount:Q", format=".1%"), alt.Tooltip("margin:Q", format=".1%")],
).add_params(pick_rep).properties(width=320, height=240, title="Vendedores, jan–set 2026 (clique num)")
labels = alt.Chart(by_rep[by_rep["discount"] > 0.1]).mark_text(dx=12, align="left", fontSize=10, color=MUTED).encode(
    x="discount:Q", y="margin:Q", text="rep_id")
cat_bars = alt.Chart(by_rep_cat).transform_filter(pick_rep).mark_bar(color=ORANGE, cornerRadius=2).encode(
    y=alt.Y("category:N", title=None, sort="-x"), x=alt.X("discount:Q", title="Desconto", axis=alt.Axis(format="%")),
    tooltip=["rep_id", "category", alt.Tooltip("discount:Q", format=".1%")],
).properties(width=260, height=240, title="Desconto por categoria do vendedor selecionado")
alt.hconcat(scatter + labels, cat_bars)

alt.HConcatChart(...)

In [8]:
r7 = by_rep.set_index("rep_id").loc["REP07"]
peers = by_rep[by_rep["rep_id"] != "REP07"][["gross", "net", "cost"]].sum()
peer_disc = 1 - peers["net"] / peers["gross"]
print(f"REP07: desconto {r7['discount']:.1%} vs {peer_disc:.1%} nos outros vendedores; "
      f"margem {r7['margin']:.1%} vs {(peers['net'] - peers['cost']) / peers['net']:.1%}")
print(f"Desconto a mais nas vendas a preço de tabela do REP07: €{r7['gross'] * (r7['discount'] - peer_disc):,.0f} em jan–set 2026")

REP07: desconto 15,1% vs 6,0% nos outros vendedores; margem 13,6% vs 21,9%
Desconto a mais nas vendas a preço de tabela do REP07: 272 794 € em jan–set 2026


**Resposta.** O REP07 dá cerca de 15% de desconto, contra 6% de todos os outros, em todas as categorias, por isso a diferença é um
hábito do vendedor e não o mix de produtos (regra R4). A faturação por cliente não é maior, portanto o desconto não compra volume.

### P3. Que clientes devemos contactar esta semana?

Dois factos orientam a lista de contactos. Primeiro, os clientes encomendam mais nas semanas a seguir a uma visita (R1). O gráfico
abaixo mede-o a partir dos dados: para cada semana de cada cliente, a parte das semanas com encomenda, conforme as semanas desde a
última visita.

In [9]:
# Painel semanal: o cliente encomendou nessa semana, e há quantas semanas foi a última visita?
wk = lambda s: s.dt.to_period("W-SUN").dt.start_time
ow = orders.assign(week=wk(orders["order_date"]))[["customer_id", "week"]].drop_duplicates().assign(ordered=1)
vw = visits.assign(week=wk(visits["visit_date"]))[["customer_id", "week"]].drop_duplicates()
weeks = pd.date_range("2024-03-04", "2026-09-21", freq="W-MON")
panel = pd.MultiIndex.from_product([customers["customer_id"], weeks], names=["customer_id", "week"]).to_frame(index=False)
panel = pd.merge_asof(panel.sort_values("week"), vw.rename(columns={"week": "last_visit"}).sort_values("last_visit"),
                      left_on="week", right_on="last_visit", by="customer_id", direction="backward")
panel["weeks_since_visit"] = ((panel["week"] - panel["last_visit"]).dt.days // 7).clip(upper=16)
panel = panel.merge(ow, on=["customer_id", "week"], how="left").fillna({"ordered": 0})
rate = panel.dropna(subset=["weeks_since_visit"]).groupby("weeks_since_visit")["ordered"].mean().reset_index()

alt.Chart(rate).mark_line(point=True, color=BLUE).encode(
    x=alt.X("weeks_since_visit:Q", title="Semanas desde a última visita (16 = 16 ou mais)"),
    y=alt.Y("ordered:Q", title="Semanas-cliente com encomenda", axis=alt.Axis(format="%")),
    tooltip=[alt.Tooltip("weeks_since_visit:Q", title="Semanas"), alt.Tooltip("ordered:Q", title="Taxa de encomenda", format=".1%")],
).properties(width=560, height=220, title="Quanto mais tempo sem visita, menos os clientes encomendam")

alt.Chart(...)

Segundo, a lista propriamente dita: clientes sem visita há mais de 60 dias na data de referência, ordenados pelo que
compraram nos últimos 12 meses. **Arraste sobre o histograma** para escolher um intervalo de dias; as barras da direita mostram
a faturação a 12 meses dos clientes nesse intervalo, por segmento.

In [10]:
last_visit = visits.groupby("customer_id")["visit_date"].max()
l12m = sales[sales["order_date"] > AS_OF - pd.Timedelta(days=365)].groupby("customer_id")["net"].sum()
cust = customers.set_index("customer_id")[["segment", "region", "rep_id"]].copy()
cust["days_since_visit"] = (AS_OF - last_visit).dt.days
cust["revenue_l12m"] = l12m
cust = cust.fillna({"revenue_l12m": 0}).reset_index()
call_list = cust[cust["days_since_visit"] > 60].sort_values("revenue_l12m", ascending=False)
print(f"{len(call_list)} clientes sem visita há 60+ dias; compraram €{call_list['revenue_l12m'].sum() / 1e6:,.1f}m nos últimos 12 meses")

brush = alt.selection_interval(encodings=["x"], value={"x": [60, 200]})
hist = alt.Chart(cust).mark_bar(color=BLUE).encode(
    x=alt.X("days_since_visit:Q", bin=alt.Bin(step=15), title="Dias desde a última visita, a 30 set 2026"),
    y=alt.Y("count():Q", title="Clientes"),
    color=alt.condition(brush, alt.value(ORANGE), alt.value(BLUE)),
).add_params(brush).properties(width=360, height=220, title="Arraste para escolher um intervalo")
seg = alt.Chart(cust).transform_filter(brush).mark_bar(color=ORANGE, cornerRadius=2).encode(
    y=alt.Y("segment:N", title=None, sort="-x"),
    x=alt.X("sum(revenue_l12m):Q", title="Comprado nos últimos 12 meses (€)", axis=alt.Axis(format="~s")),
    tooltip=["segment:N", alt.Tooltip("count():Q", title="Clientes"),
             alt.Tooltip("sum(revenue_l12m):Q", title="Faturação a 12 meses", format=",.0f")],
).properties(width=280, height=220, title="Clientes no intervalo, por segmento")
alt.hconcat(hist, seg)

225 clientes sem visita há 60+ dias; compraram 5,4 M€ nos últimos 12 meses


alt.HConcatChart(...)

In [11]:
call_list.head(10)[["customer_id", "segment", "region", "rep_id", "days_since_visit", "revenue_l12m"]]

,customer_id,segment,region,rep_id,days_since_visit,revenue_l12m
1007,C1008,Clínica veterinária,Lisboa e Vale do Tejo,REP10,62,"310,186.41"
79,C0080,Clínica veterinária,Norte,REP03,61,"183,717.31"
150,C0151,Clínica veterinária,Lisboa e Vale do Tejo,REP09,110,"154,978.19"
146,C0147,Exploração pecuária,Centro,REP04,68,"140,847.59"
1637,C1638,Clínica veterinária,Centro,REP04,64,"111,675.73"
39,C0040,Exploração pecuária,Sul,REP11,83,"108,625.46"
1174,C1175,Clínica veterinária,Sul,REP12,113,"100,523.85"
234,C0235,Loja de animais,Lisboa e Vale do Tejo,REP10,71,"99,501.42"
357,C0358,Loja de animais,Centro,REP04,97,"90,186.02"
386,C0387,Clínica veterinária,Norte,REP02,92,"89,312.52"


**Resposta.** 225 clientes estão há mais de 60 dias sem visita e compraram 5,4 M€ nos últimos 12 meses.
Ordená-los por essa faturação dá a cada vendedor uma lista de contactos para a semana; a curva da taxa de encomenda explica porque importa.

### P4. Onde está o dinheiro parado?

Faturas em aberto na data de referência, por dias de atraso face ao vencimento. **Clique num escalão de antiguidade na legenda** para
ficar só com esse escalão.

In [12]:
inv = invoices.merge(payments[["invoice_id", "payment_date"]], on="invoice_id", how="left") \
              .merge(orders[["order_id", "customer_id"]], on="order_id").merge(customers[["customer_id", "segment"]], on="customer_id")
open_inv = inv[inv["payment_date"].isna()].copy()
overdue = (AS_OF - open_inv["due_date"]).dt.days
open_inv["bucket"] = pd.cut(overdue, [-10_000, 0, 30, 60, 90, 10_000], labels=["Not due", "1-30", "31-60", "61-90", "90+"])
aging = open_inv.groupby(["segment", "bucket"], observed=True)["amount_eur"].sum().reset_index()

ORDER = ["Not due", "1-30", "31-60", "61-90", "90+"]
pick_b = alt.selection_point(fields=["bucket"], bind="legend")
alt.Chart(aging).mark_bar().encode(
    y=alt.Y("segment:N", title=None, sort="-x"),
    x=alt.X("sum(amount_eur):Q", title="Valores a receber em aberto (€, com IVA)", axis=alt.Axis(format="~s")),
    color=alt.Color("bucket:N", sort=ORDER, title="Dias de atraso",
                    scale=alt.Scale(domain=ORDER, range=["#cfe0f5", "#a8c8ef", "#7aaee6", "#3f86d8", ORANGE])),
    order=alt.Order("bucket_order:Q"),
    opacity=alt.condition(pick_b, alt.value(1), alt.value(0.15)),
    tooltip=["segment", "bucket", alt.Tooltip("amount_eur:Q", format=",.0f")],
).transform_calculate(bucket_order=f"indexof({ORDER}, datum.bucket)").add_params(pick_b).properties(
    width=560, height=200, title="Valores a receber em aberto por segmento e antiguidade (clique na legenda)")

alt.Chart(...)

In [13]:
od90 = aging[aging["bucket"] == "90+"].set_index("segment")["amount_eur"]
billed = inv.groupby("segment")["amount_eur"].sum()
paid = inv.dropna(subset=["payment_date"])
late = (paid["payment_date"] - paid["due_date"]).dt.days.groupby(paid["segment"]).mean()
pd.DataFrame({"share_of_billing": billed / billed.sum(), "share_of_90plus": od90 / od90.sum(),
              "avg_days_late": late}).sort_values("share_of_90plus", ascending=False)

,share_of_billing,share_of_90plus,avg_days_late
segment,,,
Exploração pecuária,0.17,0.84,29.15
Clínica veterinária,0.45,0.08,3.31
Loja de animais,0.25,0.07,3.32
Loja agrícola,0.13,0.01,3.24


**Resposta.** As explorações pecuárias são 17% da faturação, mas 84% do que está vencido há mais de 90 dias, e pagam em média 29 dias
depois do vencimento, contra cerca de 3 dias para todos os outros (regra R5). Condições de crédito ou um adiantamento para as explorações
libertariam a maior parte do dinheiro parado.

### P5. O que é sazonal?

Índice sazonal = faturação média mensal da categoria nesse mês do calendário ÷ o seu mês médio, nos anos completos de
2024 e 2025. **Clique numa categoria na legenda** para a destacar.

In [14]:
full = sales[sales["order_date"].dt.year.isin([2024, 2025])]
season = full.groupby(["category", full["order_date"].dt.month.rename("m")])["net"].sum().div(2).reset_index()
season["index"] = season["net"] / season.groupby("category")["net"].transform("mean")
season["month"] = pd.to_datetime(dict(year=2025, month=season["m"], day=1))

pick_c = alt.selection_point(fields=["category"], bind="legend", value=[{"category": "Vaccines"}])
alt.Chart(season).mark_line(point=True).encode(
    x=alt.X("month:T", title=None, axis=alt.Axis(format="%b")),
    y=alt.Y("index:Q", title="Índice sazonal (1,0 = mês médio)", scale=alt.Scale(zero=False)),
    color=alt.Color("category:N", title="Categoria (clique)",
                    scale=alt.Scale(domain=["Vaccines", "Parasiticides", "Pharmaceuticals", "Pet food", "Hygiene", "Accessories"],
                                    range=[BLUE, ORANGE, GREEN, "#9a6fd0", "#c2a34a", MUTED])),
    opacity=alt.condition(pick_c, alt.value(1), alt.value(0.12)),
    tooltip=["category", alt.Tooltip("month:T", format="%B"), alt.Tooltip("index:Q", format=".2f")],
).add_params(pick_c).properties(width=560, height=260, title="Índice sazonal por categoria, 2024–2025")

alt.Chart(...)

In [15]:
avg = lambda ms: season[season["m"].isin(ms)].groupby("category")["net"].mean()
pd.DataFrame({"spring_vs_winter": avg([3, 4, 5]) / avg([12, 1, 2]),
              "summer_vs_winter": avg([6, 7, 8]) / avg([12, 1, 2])}).sort_values("summer_vs_winter", ascending=False)

,spring_vs_winter,summer_vs_winter
category,,
Antiparasitários,1.61,2.34
Vacinas,1.99,1.09
Alimentação,0.99,0.99
Higiene,0.97,0.98
Acessórios,0.97,0.97
Medicamentos,0.97,0.97


**Resposta.** As vacinas vendem por mês o dobro na primavera do que no inverno, e os antiparasitários 2,3 vezes mais no verão
(regra R6); as outras categorias são estáveis. O stock e os objetivos dos vendedores para essas duas devem seguir a época.

## Passo 5. Conferir o pandas com os marts do dbt

Duas implementações independentes das mesmas definições, SQL no dbt e pandas aqui, têm de dar os mesmos números.
Se não derem, uma delas tem um erro. Esta célula lê os marts que o `dbt build` escreveu no DuckDB.

In [16]:
db = duckdb.connect(str(DATA / "commercial.duckdb"), read_only=True)
m_region = db.sql("select region, revenue_yoy_pct, visits_yoy_pct from mart_region_ytd").df().set_index("region")
m_rep = db.sql("select rep_id, discount_pct_ytd, margin_pct_ytd from mart_rep_pricing").df().set_index("rep_id")
m_call = db.sql("select count(*) n, sum(revenue_l12m) v from mart_call_list").df().iloc[0]
m_pay = db.sql("select segment, overdue_90_share, avg_days_late from mart_payment_by_segment").df().set_index("segment")
m_season = db.sql("select category, spring_vs_winter, summer_vs_winter from mart_category_seasons").df().set_index("category")
db.close()

q1i = q1.set_index("region")
rep_cur = by_rep.set_index("rep_id")
farm = "Livestock farm"
recon = pd.DataFrame([
    ["Q1 Sul revenue change", q1i.loc["Sul", "revenue_change"], m_region.loc["Sul", "revenue_yoy_pct"]],
    ["Q1 Sul visits change", q1i.loc["Sul", "visits_change"], m_region.loc["Sul", "visits_yoy_pct"]],
    ["Q2 REP07 discount", rep_cur.loc["REP07", "discount"], m_rep.loc["REP07", "discount_pct_ytd"]],
    ["Q2 REP07 margin", rep_cur.loc["REP07", "margin"], m_rep.loc["REP07", "margin_pct_ytd"]],
    ["Q3 customers on the call list", len(call_list), m_call["n"]],
    ["Q3 their 12-month revenue (€)", call_list["revenue_l12m"].sum(), m_call["v"]],
    ["Q4 farms' share of 90+ overdue", (od90 / od90.sum())[farm], m_pay.loc[farm, "overdue_90_share"]],
    ["Q4 farms' average days late", late[farm], m_pay.loc[farm, "avg_days_late"]],
    ["Q5 vaccines, spring vs winter", (avg([3, 4, 5]) / avg([12, 1, 2]))["Vaccines"], m_season.loc["Vaccines", "spring_vs_winter"]],
    ["Q5 parasiticides, summer vs winter", (avg([6, 7, 8]) / avg([12, 1, 2]))["Parasiticides"], m_season.loc["Parasiticides", "summer_vs_winter"]],
], columns=["measure", "pandas", "dbt"])
recon["match"] = np.isclose(recon["pandas"], recon["dbt"], rtol=1e-6)
assert recon["match"].all(), recon
recon

,measure,pandas,dbt,match
0,Q1 Sul revenue change,-0.16,-0.16,True
1,Q1 Sul visits change,-0.48,-0.48,True
2,Q2 REP07 discount,0.15,0.15,True
3,Q2 REP07 margin,0.14,0.14,True
4,Q3 customers on the call list,225.00,225.00,True
5,Q3 their 12-month revenue (€),"5,371,040.72","5,371,040.72",True
6,Q4 farms' share of 90+ overdue,0.84,0.84,True
7,Q4 farms' average days late,29.15,29.15,True
8,"Q5 vaccines, spring vs winter",1.99,1.99,True
9,"Q5 parasiticides, summer vs winter",2.34,2.34,True


As dez medidas coincidem. Os mesmos marts alimentam todas as interfaces, por isso os painéis não podem divergir.

## Passo 6. Publicar

O mesmo modelo aparece de três formas, cada uma onde um recrutador ou um utilizador o encontraria:

* **Painel web** ([`web/`](../web), online em [guilhermerigonato.com/commercial-analytics](https://guilhermerigonato.com/commercial-analytics/?lang=pt)):
  [`scripts/export.py`](../scripts/export.py) escreve pequenos ficheiros JSON a partir dos marts, e JavaScript simples desenha gráficos
  SVG. Clicar numa região, vendedor, categoria, segmento ou faixa de recência filtra os outros painéis.
* **Power BI** ([`powerbi/`](../powerbi)): um projeto PBIP sobre o mesmo esquema em estrela, com medidas DAX,
  segurança ao nível da linha por região e uma verificação do Best Practice Analyzer em [`bpa-rules.json`](../bpa-rules.json).
* **Qlik Sense** ([`qlik/`](../qlik)): o script de carga constrói o mesmo modelo; o motor associativo dá o
  filtro cruzado de origem.

## Que biblioteca usar para gráficos interativos

| Biblioteca | Gráficos ligados | Resultado | Melhor para |
|---|---|---|---|
| **Altair** (Vega-Lite), usada aqui | Nativos: seleções partilhadas entre gráficos | HTML estático, sem servidor | Relatórios e explorações que têm de abrir em qualquer lado |
| **Plotly** | Por gráfico; ligar gráficos exige Dash | HTML; uma app Dash para ligar | Gráficos individuais ricos, mapas |
| **Bokeh** | Seleção ligada através de uma fonte de dados partilhada | HTML ou um servidor Bokeh | Dispersões grandes, ferramentas à medida |
| **HoloViews + Panel** | `link_selections` numa só chamada | Notebook ou app Panel | Transformar um notebook numa pequena app |
| **Dash / Streamlit** | Qualquer, com callbacks em Python | Precisa de servidor | Ferramentas internas com dados ao vivo |